here's an awkward fact about the previous two chapters: black holes are invisible.

i'm not being metphorical here. literally, a bare black hole emits nothing, reflects nothing and is small beyond reason — chapter one's entire event horizon, for a stellar-mass hole, would fit inside a city. if black holes were tidy eaters, we would probably still be arguing about whether they exist.

luckily, they are the messiest eaters in the universe.

so this chapter is about the mess. the accretion disk - what it is, why it exists, why it glows, and how to photograph it. by the end we'll have rendered the full anatomy: the lensed arcs, the photon ring, the shadow. this part always got me in uni, so please give me feedback for improvement!

(we're going back to the non-spinning schwarzschild hole for this one, to keep the maths friendly. everything here generalizes to kerr, and chapter two is sitting right there when we want it)

a question i always had was, why is there a disk at all?

a natural first guess is that stuff near a black hole just... falls in. straight down, like a rock off a cliff.

but almost nothing in the universe is aimed straight at anything. gas clouds, stripped stars all arrive with angular momentum and chapter one covers exactly what angular momentum does near a black hole, it puts a centrifugal barrier between you and the horizon. you don't fall in- you orbit.

i think a helpful way to visulaise this is the earlier drain analogy? water doesn't plummet down a drain either. it circles, crowds inward and forms a swirling sheet. in the case of our black hole, collisions flatten the infalling matter into a thin spinning disk, and then something cool happens,

friction.

adjacent rings of the disk orbit at different speeds (inner rings faster because kepler says so), so they rub against each other. the rubbing does two jobs at once,
* it bleeds away angular momentum, letting matter drift slowly inward, ring by ring, toward chapter one's last parking spot at $r = 6M$ and then over the edge
* it converts orbital energy into insane amounts of heat. the inner disk reaches millions of degrees and glows in x-rays

that is,

> the black hole does not shine. the disk lights itself.

every photon in our render will be gravitational potential energy, cashed out as heat by friction, on the way down. the black hole's only contribution is the plumbing.

now, how do we image this>

firing photons off the disk and hoping some fly into our camera is a lottery - almost all of them miss. so instead, for every pixel of a virtual camera, we ask, if light arrived here, where did it come from? we launch the ray backwards into the curved spacetime and integrate until it,

1. hits the disk -> shade the pixel with the disk's glow
2. crosses the horizon -> black pixel. that's the shadow
3. escapes to infinity -> black background

every pixel gets an answer.

and for the geodesics themselves, every photon path in schwarzschild lies in a plane, and writing $u = 1/r$ as a function of the swept angle $\phi$, null geodesics obey the almost offensively compact binet equation,

$$\frac{d^2u}{d\phi^2} + u = 3Mu^2$$

if we look at it really closely. the left side alone gives straight lines - that's flat space (newton's optics). the right side is our friend from chapter one, the einstein term, back for an encore. light bending, the photon sphere at $3M$, the shadow at $b_c = 3\sqrt3\,M$ all of it lives in that one little $3Mu^2$.

simulation time!!

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time

# geometrized units, G = c = 1. quantum physicists know where the door is
M = 1.0
# event horizon
R_S = 2 * M     
# the disk: from the last parking spot (ISCO) to an outer edge             
R_IN, R_OUT = 6 * M, 18 * M  

plt.rcParams['figure.dpi'] = 110

we're going to assume our camera is a pinhole camera hovering at $r = 45M$, tilted $82°$ from the disk axis so we see the disk nearly edge on. each pixel defines a direction and each direction becomes a ray.

In [2]:
def camera_rays(width, height, r_cam=45.0, incl_deg=82.0, fov_deg=17.0):
    """camera position plus one unit direction per pixel, shape (W*H, 3)."""
    th = np.radians(incl_deg)
    cam = np.array([np.sin(th), 0.0, np.cos(th)]) * r_cam
    # look at the origin
    fwd = -cam / np.linalg.norm(cam)                    
    right = np.cross(fwd, [0.0, 0.0, 1.0]); right /= np.linalg.norm(right)
    up = np.cross(right, fwd)
    tanf = np.tan(np.radians(fov_deg) / 2)
    xs = np.linspace(-1, 1, width) * tanf
    ys = np.linspace(-1, 1, height) * tanf * (height / width)
    X, Y = np.meshgrid(xs, ys)
    d = fwd[None, None, :] + X[..., None] * right + Y[..., None] * up
    d /= np.linalg.norm(d, axis=-1, keepdims=True)
    return cam, d.reshape(-1, 3)

